## Risk-Based Analysis

This notebook analyzes the likelihood of high-severity accidents under different conditions.
Focus is on conditional probability rather than simple distributions.

### Load Cleaned Dataset

In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

df = pd.read_csv(DATA_PROCESSED / "accidents_cleaned.csv")

print(df.shape)
df.head()

(192487, 17)


,Severity,Start_Time,Start_Lat,Start_Lng,Visibility(mi),Weather_Condition,Junction,Traffic_Signal,year,month,hour,day_of_week,high_severity,night,peak_hour,adverse_weather,weather_group
0,1,2020-04-17 09:29:30,26.706900,-80.119360,10.0,Mostly Cloudy,False,True,2020.0,4.0,9.0,4.0,0,0,1,0,Cloudy
1,3,2019-09-20 15:22:16,47.118706,-122.556908,10.0,Cloudy,False,False,2019.0,9.0,15.0,4.0,1,0,0,0,Cloudy
2,2,2019-06-03 16:55:43,33.451355,-111.890343,10.0,Fair,False,False,2019.0,6.0,16.0,0.0,0,0,1,0,Clear/Fair
3,2,2021-02-04 12:48:21,42.448910,-93.721138,0.5,Light Snow / Windy,False,False,2021.0,2.0,12.0,3.0,0,0,0,1,Snow/Ice
4,2,2022-06-23 10:57:30,38.858169,-77.219815,10.0,Cloudy,False,False,2022.0,6.0,10.0,3.0,0,0,0,0,Cloudy


### Overall Severity Risk

Baseline probability of high severity across all accidents.

In [2]:
baseline_risk = df["high_severity"].mean()
print("Baseline high severity risk:", baseline_risk)

Baseline high severity risk: 0.15601573093247856


### Risk by Individual Conditions

Analyzing how risk varies across single variables.

In [3]:
df.groupby("night")["high_severity"].mean()

night
0    0.160427
1    0.139498
Name: high_severity, dtype: float64

In [4]:
df.groupby("peak_hour")["high_severity"].mean()

peak_hour
0    0.152510
1    0.160623
Name: high_severity, dtype: float64

In [5]:
df.groupby("weather_group")["high_severity"].mean().sort_values(ascending=False)

weather_group
Rain             0.199524
Storm            0.191313
Snow/Ice         0.179735
Cloudy           0.170779
Unknown          0.157831
Other            0.144781
Clear/Fair       0.137693
Fog/Mist/Haze    0.116452
Name: high_severity, dtype: float64

In [6]:
df.groupby("adverse_weather")["high_severity"].mean()

adverse_weather
0    0.152778
1    0.182397
Name: high_severity, dtype: float64

In [7]:
df.groupby("hour")["high_severity"].mean()

hour
0.0     0.104081
1.0     0.092004
2.0     0.096617
3.0     0.125258
4.0     0.166954
5.0     0.156067
6.0     0.158724
7.0     0.146981
8.0     0.148879
9.0     0.172602
10.0    0.169161
11.0    0.170523
12.0    0.163147
13.0    0.154980
14.0    0.151802
15.0    0.158591
16.0    0.157112
17.0    0.158778
18.0    0.179566
19.0    0.179041
20.0    0.168994
21.0    0.163974
22.0    0.149164
23.0    0.098103
Name: high_severity, dtype: float64

### Risk Difference from Baseline

Comparing each condition’s risk against overall baseline.

In [9]:
risk_by_weather = df.groupby("weather_group")["high_severity"].mean()

risk_diff = risk_by_weather - baseline_risk

pd.DataFrame({
    "risk": risk_by_weather,
    "difference_from_baseline": risk_diff
}).sort_values("difference_from_baseline", ascending=False)

,risk,difference_from_baseline
weather_group,,
Rain,0.199524,0.043508
Storm,0.191313,0.035298
Snow/Ice,0.179735,0.023720
Cloudy,0.170779,0.014764
Unknown,0.157831,0.001815
Other,0.144781,-0.011235
Clear/Fair,0.137693,-0.018322
Fog/Mist/Haze,0.116452,-0.039564


### Risk Under Combined Conditions

Analyzing how combinations of conditions affect severity risk.

In [10]:
df.groupby(["night", "adverse_weather"])["high_severity"].mean()

night  adverse_weather
0      0                  0.156976
       1                  0.189530
1      0                  0.136774
       1                  0.159142
Name: high_severity, dtype: float64

In [11]:
df.groupby(["night", "adverse_weather"])["high_severity"].mean()


night  adverse_weather
0      0                  0.156976
       1                  0.189530
1      0                  0.136774
       1                  0.159142
Name: high_severity, dtype: float64

In [12]:
df.groupby(["night", "weather_group"])["high_severity"].mean()

night  weather_group
0      Clear/Fair       0.141913
       Cloudy           0.173912
       Fog/Mist/Haze    0.129362
       Other            0.146119
       Rain             0.201060
       Snow/Ice         0.190431
       Storm            0.188457
       Unknown          0.170043
1      Clear/Fair       0.123135
       Cloudy           0.157063
       Fog/Mist/Haze    0.078806
       Other            0.141026
       Rain             0.193874
       Snow/Ice         0.148423
       Storm            0.211864
       Unknown          0.125000
Name: high_severity, dtype: float64

In [13]:
df.groupby(["hour", "weather_group"])["high_severity"].mean()

hour  weather_group
0.0   Clear/Fair       0.103427
      Cloudy           0.118744
      Fog/Mist/Haze    0.027027
      Other            0.000000
      Rain             0.111650
                         ...   
23.0  Other            0.142857
      Rain             0.145749
      Snow/Ice         0.139535
      Storm            0.100000
      Unknown          0.034783
Name: high_severity, Length: 192, dtype: float64

### High-Risk Scenarios

Identifying conditions with highest probability of severe accidents.

In [14]:
scenario_risk = df.groupby(
    ["night", "adverse_weather", "peak_hour"]
)["high_severity"].mean().reset_index()

scenario_risk.sort_values(by="high_severity", ascending=False).head(10)

,night,adverse_weather,peak_hour,high_severity
2,0,1,0,0.196323
3,0,1,1,0.184137
5,1,1,0,0.159142
1,0,0,1,0.157778
0,0,0,0,0.156010
4,1,0,0,0.136774


In [15]:
OUTPUTS = PROJECT_ROOT / "outputs" / "tables"
OUTPUTS.mkdir(parents=True, exist_ok=True)

scenario_risk.to_csv(OUTPUTS / "high_risk_scenarios.csv", index=False)